# Multimodal Crop Health Monitoring — Setup & EDA

**Author:** Khang Phan  
**Purpose:** Environment verification, dataset loading, and exploratory data analysis.

This notebook:
1. Checks that all dependencies are installed
2. Loads the real crop yield CSVs from `../data/raw/`
3. Produces summary statistics and visualizations

## 1. Environment Check

In [ ]:
import importlib, sys

packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "torch": "torch",
    "torchvision": "torchvision",
    "cv2": "opencv-python",
    "streamlit": "streamlit",
    "seaborn": "seaborn",
    "PIL": "pillow",
}

all_ok = True
for module, pip_name in packages.items():
    try:
        mod = importlib.import_module(module)
        version = getattr(mod, "__version__", "installed")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} → pip install {pip_name}")
        all_ok = False

print(f"\nPython {sys.version}")
print("\nEnvironment ready!" if all_ok else "\nInstall missing packages, then re-run.")

## 2. Available Datasets

The following CSVs are in `../data/raw/`:

| File | Description |
|---|---|
| `yield_df.csv` | Main dataset — yield + rainfall + temp + pesticides combined (28,242 rows) |
| `yield.csv` | Raw yield by country, crop, year |
| `rainfall.csv` | Average annual rainfall by country |
| `temp.csv` | Average temperature by country |
| `pesticides.csv` | Pesticide usage by country and year |

In [ ]:
import pandas as pd
from pathlib import Path

DATA_RAW = Path("../data/raw")

# List all available CSVs
csv_files = sorted(DATA_RAW.glob("*.csv"))
print("CSVs found in data/raw/:")
for f in csv_files:
    df_tmp = pd.read_csv(f, nrows=0)
    print(f"  {f.name:<25} columns: {list(df_tmp.columns)}")

## 3. Crop Yield Dataset — Load & Explore

Loading `yield_df.csv` — the combined dataset with yield, rainfall, temperature, and pesticide columns.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_RAW = Path("../data/raw")

df = pd.read_csv(DATA_RAW / "yield_df.csv", index_col=0)

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nUnique crops : {df['Item'].nunique()}")
print(f"Unique regions: {df['Area'].nunique()}")
print(f"Year range   : {df['Year'].min()} – {df['Year'].max()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head(10)

In [ ]:
print("Summary statistics for numeric columns:")
df.describe().round(2)

## 4. Yield Dataset — Visualizations

In [ ]:
import matplotlib.pyplot as plt
import pathlib

RESULTS = pathlib.Path("../results")
RESULTS.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Crop Yield Dataset — Exploratory Analysis", fontsize=15, fontweight="bold")

# 1. Yield distribution
axes[0, 0].hist(df["hg/ha_yield"], bins=40, color="#4CAF50", edgecolor="white", alpha=0.85)
axes[0, 0].set_title("Crop Yield Distribution")
axes[0, 0].set_xlabel("Yield (hg/ha)")
axes[0, 0].set_ylabel("Count")

# 2. Average yield by crop
if "Item" in df.columns:
    crop_yield = df.groupby("Item")["hg/ha_yield"].mean().sort_values(ascending=False)
    axes[0, 1].barh(crop_yield.index, crop_yield.values, color="#2196F3", alpha=0.85)
    axes[0, 1].set_title("Average Yield by Crop")
    axes[0, 1].set_xlabel("Mean Yield (hg/ha)")

# 3. Rainfall vs Yield scatter
axes[1, 0].scatter(df["average_rain_fall_mm_per_year"], df["hg/ha_yield"],
                   alpha=0.3, s=12, color="#FF9800")
axes[1, 0].set_title("Rainfall vs Yield")
axes[1, 0].set_xlabel("Avg Rainfall (mm/year)")
axes[1, 0].set_ylabel("Yield (hg/ha)")

# 4. Temperature vs Yield scatter
axes[1, 1].scatter(df["avg_temp"], df["hg/ha_yield"],
                   alpha=0.3, s=12, color="#9C27B0")
axes[1, 1].set_title("Temperature vs Yield")
axes[1, 1].set_xlabel("Avg Temperature (°C)")
axes[1, 1].set_ylabel("Yield (hg/ha)")

plt.tight_layout()
plt.savefig(RESULTS / "eda_yield.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved to results/eda_yield.png")

## 5. PlantVillage Image Dataset — Exploration

Counts images per class. Falls back to generating synthetic sample images if dataset is not yet downloaded.

In [ ]:
import pathlib, numpy as np, matplotlib.pyplot as plt, matplotlib.patches as mpatches
from PIL import Image

pv_dir = pathlib.Path("../data/raw/PlantVillage")

if pv_dir.exists():
    # Real dataset: count images per class folder
    class_counts = {d.name: len(list(d.glob("*.jpg")) + list(d.glob("*.JPG")))
                    for d in sorted(pv_dir.iterdir()) if d.is_dir()}
    use_synthetic = False
    print(f"PlantVillage: {len(class_counts)} classes, {sum(class_counts.values()):,} total images")
else:
    # Synthetic class distribution mimicking PlantVillage
    use_synthetic = True
    print("PlantVillage not found — using representative synthetic class distribution.\n")
    class_counts = {
        "Tomato_healthy": 1591, "Tomato_Early_blight": 1000, "Tomato_Late_blight": 1909,
        "Tomato_Leaf_Mold": 952,  "Tomato_Septoria_leaf_spot": 1771,
        "Potato_healthy": 152,   "Potato_Early_blight": 1000, "Potato_Late_blight": 1000,
        "Corn_healthy": 1162,    "Corn_Common_rust": 1192,    "Corn_Northern_Leaf_Blight": 985,
        "Apple_healthy": 1645,   "Apple_scab": 630,           "Apple_Black_rot": 621,
        "Grape_healthy": 423,    "Grape_Black_rot": 1180,     "Grape_Leaf_blight": 1076,
        "Pepper_healthy": 1478,  "Pepper_Bacterial_spot": 997,
    }

# Plot class distribution (top 15)
top = dict(sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:15])
colors = ["#4CAF50" if "healthy" in k.lower() else "#F44336" for k in top]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(list(top.keys()), list(top.values()), color=colors, alpha=0.85, edgecolor="white")
ax.set_xlabel("Number of Images")
ax.set_title("PlantVillage — Top 15 Classes by Image Count\n(green = healthy, red = disease)",
             fontweight="bold")
ax.bar_label(bars, padding=3, fontsize=8)
plt.tight_layout()
plt.savefig("../results/eda_plantvillage.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Total classes shown: {len(top)} | Plot saved to results/eda_plantvillage.png")

## 6. Sample Image Grid (PlantVillage)

Displays a 3×3 grid of sample leaf images — real if available, otherwise synthetic color patches for layout verification.

In [ ]:
import random, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

pv_dir = Path("../data/raw/PlantVillage")
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle("Sample Leaf Images — PlantVillage Dataset", fontsize=13, fontweight="bold")

all_images = list(pv_dir.rglob("*.jpg")) + list(pv_dir.rglob("*.JPG")) if pv_dir.exists() else []

for i, ax in enumerate(axes.flat):
    if all_images:
        img_path = random.choice(all_images)
        img = Image.open(img_path).resize((224, 224))
        ax.imshow(img)
        label = img_path.parent.name.replace("_", "\n")
    else:
        # Synthetic placeholder: colored square with label
        color = np.random.uniform(0.1, 0.9, (224, 224, 3))
        color[:, :, 1] = color[:, :, 1] * 0.6 + 0.2  # greenish tint
        ax.imshow(color)
        labels = ["Tomato_healthy", "Tomato_Early_blight", "Potato_Late_blight",
                  "Corn_healthy", "Apple_scab", "Grape_Black_rot",
                  "Pepper_healthy", "Corn_Common_rust", "Potato_healthy"]
        label = labels[i].replace("_", "\n")
    ax.set_title(label, fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.savefig("../results/sample_images.png", dpi=100, bbox_inches="tight")
plt.show()
note = "(synthetic placeholders)" if not all_images else "(real PlantVillage images)"
print(f"Image grid {note} saved to results/sample_images.png")

## 7. Correlation Heatmap — Yield Features

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_cols = ["hg/ha_yield", "average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "Year"]
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels([c.replace("_", "\n") for c in num_cols], fontsize=8)
ax.set_yticklabels([c.replace("_", "\n") for c in num_cols], fontsize=8)

for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)

ax.set_title("Feature Correlation Matrix — Crop Yield Data", fontweight="bold")
plt.tight_layout()
plt.savefig("../results/correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print("Heatmap saved to results/correlation_heatmap.png")

## 8. Setup Complete

All outputs saved to `../results/`. Summary:

| Output | Description |
|---|---|
| `eda_yield.png` | Yield distribution, average by crop, rainfall/temp scatter |
| `eda_plantvillage.png` | PlantVillage class distribution (runs when image data is present) |
| `sample_images.png` | 3×3 leaf image grid (runs when image data is present) |
| `correlation_heatmap.png` | Feature correlation matrix |

**Next steps:**
- Run `src/yield_predictor.py` to train the XGBoost yield model
- Download PlantVillage images, then run `src/disease_classifier.py`
- Launch the dashboard: `streamlit run ui/app.py`

---

## 9. Yield Prediction — Data Preprocessing

Encode categorical columns and split into train/test sets.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv(Path("../data/raw/yield_df.csv"), index_col=0)

# Encode categorical columns
le_area = LabelEncoder()
le_item = LabelEncoder()
df["Area_enc"] = le_area.fit_transform(df["Area"])
df["Item_enc"] = le_item.fit_transform(df["Item"])

FEATURES = ["Area_enc", "Item_enc", "Year",
            "average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp"]
TARGET = "hg/ha_yield"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"\nFeatures: {FEATURES}")
print(f"Target  : {TARGET}")
X_train.head()

## 10. Train Baseline — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_r2  = r2_score(y_test, rf_preds)

print(f"Random Forest")
print(f"  MAE : {rf_mae:,.0f} hg/ha")
print(f"  R²  : {rf_r2:.4f}")

## 11. Train Main Model — XGBoost

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    verbosity=0,
)
xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_r2  = r2_score(y_test, xgb_preds)

print(f"XGBoost")
print(f"  MAE : {xgb_mae:,.0f} hg/ha")
print(f"  R²  : {xgb_r2:.4f}")

print(f"\nImprovement over Random Forest")
print(f"  MAE  delta: {rf_mae - xgb_mae:+,.0f} hg/ha")
print(f"  R²   delta: {xgb_r2 - rf_r2:+.4f}")

## 12. Evaluation — Predicted vs Actual & Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("XGBoost Yield Predictor — Evaluation", fontsize=13, fontweight="bold")

# 1. Predicted vs Actual
axes[0].scatter(y_test, xgb_preds, alpha=0.3, s=10, color="#2196F3")
lims = [min(y_test.min(), xgb_preds.min()), max(y_test.max(), xgb_preds.max())]
axes[0].plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual Yield (hg/ha)")
axes[0].set_ylabel("Predicted Yield (hg/ha)")
axes[0].set_title(f"Predicted vs Actual  (R² = {xgb_r2:.3f})")
axes[0].legend()

# 2. Feature importance
feat_names = ["Region", "Crop Type", "Year", "Rainfall", "Pesticides", "Temperature"]
importances = xgb_model.feature_importances_
order = np.argsort(importances)
axes[1].barh([feat_names[i] for i in order], importances[order], color="#4CAF50", alpha=0.85)
axes[1].set_xlabel("Importance Score")
axes[1].set_title("Feature Importance")

plt.tight_layout()
plt.savefig(RESULTS / "yield_model_evaluation.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved to results/yield_model_evaluation.png")

## 13. Save Model

In [ ]:
import pickle

save_path = RESULTS / "yield_model.pkl"
with open(save_path, "wb") as f:
    pickle.dump({
        "model": xgb_model,
        "le_area": le_area,
        "le_item": le_item,
        "features": FEATURES,
    }, f)

print(f"Model saved to {save_path}")
print(f"  MAE : {xgb_mae:,.0f} hg/ha")
print(f"  R²  : {xgb_r2:.4f}")